# LiteRT-LM kernel 0112 on a Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

Runs the captured LiteRT-LM WebGPU matrix-vector kernel 0112 (Gemma 4 E2B web, @litert-lm/core 0.17.1) verbatim (or, where Chrome offers no shader-f16, an f32 transcription of it, which the driver announces) and with deeper K splits, through headless Chrome on the T4, as `node microbench.mjs` does on the Mac. Cell 1 installs the NVIDIA Vulkan userspace Chrome needs (zero-tvm's recipe, after Chrome's colab-headless guide); if it prints `llvmpipe` instead of `Tesla T4`, stop and report the driver version from `nvidia-smi`.

Nothing is downloaded from Hugging Face: the page generates its own random weights.

Kernel 0112 is emitted at run time by `@litert-lm/core` 0.17.1 (Google LLC) and is reproduced here under the Apache License 2.0; provenance and the license text are in `out/shaders/NOTICE` and `out/shaders/LICENSE` in the repository, https://github.com/abgnydn/litert-capture, which also holds the method, the harness and the records. Independent work, not affiliated with Google.

In [ ]:
# 1) Node 22 + the NVIDIA Vulkan userspace matching the driver
import subprocess
drv = subprocess.check_output('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True).decode().split('.')[0].strip()
print('NVIDIA driver major:', drv, '-> installing libnvidia-gl-' + drv)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq update > /dev/null
# libasound2 is libasound2t64 on Colab's Ubuntu 24.04; one missing package name aborts the whole install
!apt-get -qq install -y nodejs vulkan-tools libnvidia-gl-{drv} libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2t64 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!rm -f /usr/share/vulkan/icd.d/lvp_icd*.json
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Unpack the page, the captured kernel and the driver; install puppeteer 25.11.0 (downloads Chrome for Testing 153, which made the committed T4 record)
import base64, os, json
files = {"microbench.html":"...","out/shaders/0112_unlabeled.wgsl":"ZW5hYmxlIGYxNjsKQGlkKDApIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1g6IGkzMjsKQGlkKDEpIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1k6IGkzMjsKQGlkKDIpIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1o6IGkzMjsKCmZuIFBhY2s0eDE2ZmxvYXQodiA6IHZlYzQ8ZjMyPikgLT4gdmVjMjx1MzI+IHsKICByZXR1cm4gdmVjMjx1MzI+KHBhY2syeDE2ZmxvYXQodi54eSksIHBhY2syeDE2ZmxvYXQodi56dykpOwp9CgpmbiBVbnBhY2s0eDE2ZmxvYXQodiA6IHZlYzI8dTMyPikgLT4gdmVjNDxmMzI+IHsKICByZXR1cm4gdmVjNDxmMzI+KHVucGFjazJ4MTZmbG9hdCh2LngpLCB1bnBhY2syeDE2ZmxvYXQodi55KSk7Cn0KQGdyb3VwKDApIEBiaW5kaW5nKDApIHZhciBkc3RfdGVuc29yX2ltYWdlMmQgOiB0ZXh0dXJlX3N0b3JhZ2VfMmQ8cmdiYTE2ZmxvYXQsIHdyaXRlPjsKQGdyb3VwKDApIEBiaW5kaW5nKDEpIHZhciBzcmNfdGVuc29yX2ltYWdlMmQgOiB0ZXh0dXJlXzJkPGYzMj47CkBncm91cCgwKSBAYmluZGluZygyKSB2YXIgd2VpZ2h0c19pbWFnZTJkIDogdGV4dHVyZV8yZDx1MzI+OwpzdHJ1Y3Qgd2VpZ2h0c19zY2FsZV9idWZmZXJfdmVjdG9yIHsKICBkYXRhOiBhcnJheTx2ZWM0PGYxNj4+LAp9OwpAZ3JvdXAoMCkgQGJpbmRpbmcoMykgdmFyPHN0b3JhZ2UsIHJlYWQ+IHdlaWdodHNfc2NhbGVfYnVmZmVyIDogd2VpZ2h0c19zY2FsZV9idWZmZXJfdmVjdG9yOwpzdHJ1Y3QgU2NhbGFycyB7CiAgaTAgOiB2ZWM0PGkzMj4sCn07CkBncm91cCgwKSBAYmluZGluZyg0KSB2YXI8dW5pZm9ybT4gVTogU2NhbGFyczsKdmFyPHdvcmtncm91cD4gdGVtcCA6IGFycmF5PHZlYzQ8ZjE2PiwgNjQ+OwpAY29tcHV0ZSBAd29ya2dyb3VwX3NpemUoV09SS0dST1VQX1NJWkVfWCwgV09SS0dST1VQX1NJWkVfWSwgV09SS0dST1VQX1NJWkVfWikKZm4gbWFpbihAYnVpbHRpbihnbG9iYWxfaW52b2NhdGlvbl9pZCkgcmVzZXJ2ZWRfZ2lkIDogdmVjMzx1MzI+LApAYnVpbHRpbihsb2NhbF9pbnZvY2F0aW9uX2lkKSByZXNlcnZlZF9saWQgOiB2ZWMzPHUzMj4sCkBidWlsdGluKHdvcmtncm91cF9pZCkgcmVzZXJ2ZWRfZ3JvdXBfaWQgOiB2ZWMzPHUzMj4pIHsKICB2YXIgZHN0X3MgOiBpMzI9IGkzMihyZXNlcnZlZF9naWQueCk7CiAgdmFyIGRzdF9lbmRfc2xpY2UgOiBpMzI9IFUuaTAueDsKICB2YXIgZHN0X3Nfd2dfb2Zmc2V0IDogaTMyPSBpMzIocmVzZXJ2ZWRfZ3JvdXBfaWQueCkgKiBXT1JLR1JPVVBfU0laRV9YOwogIGlmIChkc3Rfc193Z19vZmZzZXQgPj0gZHN0X2VuZF9zbGljZSkge3JldHVybjt9CiAgdmFyIHJfc3AwIDogdmVjNDxmMTY+PSB2ZWM0PGYxNj4oMC4wLCAwLjAsIDAuMCwgMC4wKTsKICB2YXIgdGlkIDogdmVjMjxpMzI+OwogIHRpZC54ID0gaTMyKHJlc2VydmVkX2xpZC54KTsKICB0aWQueSA9IGkzMihyZXNlcnZlZF9saWQueSk7CiAgaWYgKGRzdF9zIDwgVS5pMC54KSB7CiAgdmFyIHdfc2NhbGUgOiB2ZWM0PGYxNj49IHdlaWdodHNfc2NhbGVfYnVmZmVyLmRhdGFbKGRzdF9zKV07CiAgdmFyIHdfYmlhcyA6IHZlYzQ8ZjE2Pj0gLXdfc2NhbGUgKiAodmVjNDxmMTY+KDIuMCwgMi4wLCAyLjAsIDIuMCkpOwogIGZvciAodmFyIHNyY19zIDogaTMyPSB0aWQueTsgc3JjX3MgPCBVLmkwLnk7IHNyY19zICs9IDQpIHsKICAgIHZhciB2MCA6IHZlYzQ8ZjE2Pj0gdmVjNDxmMTY+KHRleHR1cmVMb2FkKHNyY190ZW5zb3JfaW1hZ2UyZCwgdmVjMjxpMzI+KCgwKSwgKCgwKSAqIFUuaTAueSArIChzcmNfcykpKSwgMCkpOwogICAgdmFyIHcwIDogdmVjNDxmMTY+O3ZhciB3MSA6IHZlYzQ8ZjE2Pjt2YXIgdzIgOiB2ZWM0PGYxNj47dmFyIHczIDogdmVjNDxmMTY+OwogICAgdmFyIHcgOiB2ZWM0PHUzMj49IHZlYzQ8dTMyPih0ZXh0dXJlTG9hZCh3ZWlnaHRzX2ltYWdlMmQsIHZlYzI8aTMyPigoZHN0X3MpLCAoc3JjX3MpKSwgMCkpOwogICAgCiAgewogIHZhciB3dDAgOiB2ZWM0PGYxNj49IHZlYzQ8ZjE2Pih3KSAqIGYxNigwLjI1KTsKICB2YXIgd3QxIDogdmVjNDxmMTY+PSBmbG9vcih3dDApOwogIHcwLnggPSAod3QwLnggLSB3dDEueCkgKiBmMTYoNC4wKTsKICB3MS54ID0gKHd0MC55IC0gd3QxLnkpICogZjE2KDQuMCk7CiAgdzIueCA9ICh3dDAueiAtIHd0MS56KSAqIGYxNig0LjApOwogIHczLnggPSAod3QwLncgLSB3dDEudykgKiBmMTYoNC4wKTsKICB3dDAgPSB3dDEgKiBmMTYoMC4yNSk7CiAgd3QxID0gZmxvb3Iod3QwKTsKICB3MC55ID0gKHd0MC54IC0gd3QxLngpICogZjE2KDQuMCk7CiAgdzEueSA9ICh3dDAueSAtIHd0MS55KSAqIGYxNig0LjApOwogIHcyLnkgPSAod3QwLnogLSB3dDEueikgKiBmMTYoNC4wKTsKICB3My55ID0gKHd0MC53IC0gd3QxLncpICogZjE2KDQuMCk7CiAgd3QwID0gd3QxICogZjE2KDAuMjUpOwogIHd0MSA9IGZsb29yKHd0MCk7CiAgdzAueiA9ICh3dDAueCAtIHd0MS54KSAqIGYxNig0LjApOwogIHcxLnogPSAod3QwLnkgLSB3dDEueSkgKiBmMTYoNC4wKTsKICB3Mi56ID0gKHd0MC56IC0gd3QxLnopICogZjE2KDQuMCk7CiAgdzMueiA9ICh3dDAudyAtIHd0MS53KSAqIGYxNig0LjApOwogIHcwLncgPSB3dDEueDsKICB3MS53ID0gd3QxLnk7CiAgdzIudyA9IHd0MS56OwogIHczLncgPSB3dDEudzsKICB9CjsKICAgIHcwID0gZm1hKHcwLCB3X3NjYWxlLCB3X2JpYXMpOwogICAgdzEgPSBmbWEodzEsIHdfc2NhbGUsIHdfYmlhcyk7CiAgICB3MiA9IGZtYSh3Miwgd19zY2FsZSwgd19iaWFzKTsKICAgIHczID0gZm1hKHczLCB3X3NjYWxlLCB3X2JpYXMpOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLngsIHYwLngsIHYwLngsIHYwLngpLCB3MCwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLnksIHYwLnksIHYwLnksIHYwLnkpLCB3MSwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLnosIHYwLnosIHYwLnosIHYwLnopLCB3Miwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLncsIHYwLncsIHYwLncsIHYwLncpLCB3Mywgcl9zcDApOwogIH0gCiAgfSAKICB0ZW1wW3RpZC54ICogNCArIHRpZC55XSA9IHJfc3AwOwogIGZvciAodmFyIHlzdHJpZGUgOiBpMzI9IDQgLyAyOyB5c3RyaWRlID4gMDsgeXN0cmlkZSAvPSAyKSB7CiAgICB3b3JrZ3JvdXBCYXJyaWVyKCk7CiAgICBpZiAodGlkLnkgPCB5c3RyaWRlKSB7CiAgICAgIHJfc3AwICs9IHRlbXBbdGlkLnggKiA0ICsgdGlkLnkgKyB5c3RyaWRlXTsKICAgICAgdGVtcFt0aWQueCAqIDQgKyB0aWQueV0gPSByX3NwMDsKICAgIH0KICB9CiAgaWYgKGRzdF9zID49IFUuaTAueCkge3JldHVybjt9CiAgaWYgKHRpZC55ICE9IDApIHtyZXR1cm47fQogIHsKICB2YXIgcmVzX3ZhbHVlIDogdmVjNDxmMTY+PSB2ZWM0PGYxNj4ocl9zcDApOwogIHRleHR1cmVTdG9yZShkc3RfdGVuc29yX2ltYWdlMmQsIHZlYzI8aTMyPigoMCksICgoMCkgKiBVLmkwLnggKyAoZHN0X3MpKSksIHZlYzQ8ZjMyPihyZXNfdmFsdWUpKTsKICB9Cn0K","run.mjs":"Ly8gaGVhZGxlc3MgQ2hyb21lIG9uIGEgQ29sYWIgVDQsIHBlciBDaHJvbWUncyBjb2xhYi1oZWFkbGVzcyBndWlkZQppbXBvcnQgeyBjcmVhdGVTZXJ2ZXIgfSBmcm9tICdub2RlOmh0dHAnCmltcG9ydCB7IGNyZWF0ZVJlYWRTdHJlYW0sIHN0YXRTeW5jLCB3cml0ZUZpbGVTeW5jIH0gZnJvbSAnbm9kZTpmcycKaW1wb3J0IHsgam9pbiwgbm9ybWFsaXplLCBleHRuYW1lIH0gZnJvbSAnbm9kZTpwYXRoJwppbXBvcnQgcHVwcGV0ZWVyIGZyb20gJ3B1cHBldGVlcicKY29uc3QgUk9PVCA9IHByb2Nlc3MuY3dkKCkKY29uc3QgTUlNRSA9IHsgJy5odG1sJzogJ3RleHQvaHRtbDsgY2hhcnNldD11dGYtOCcsICcud2dzbCc6ICd0ZXh0L3BsYWluJyB9CmNvbnN0IHNlcnZlciA9IGNyZWF0ZVNlcnZlcigocmVxLCByZXMpID0+IHsKICBjb25zdCBmID0gam9pbihST09ULCBub3JtYWxpemUoZGVjb2RlVVJJQ29tcG9uZW50KG5ldyBVUkwocmVxLnVybCwgJ2h0dHA6Ly94JykucGF0aG5hbWUpKSkKICBsZXQgczsgdHJ5IHsgcyA9IHN0YXRTeW5jKGYpLnNpemUgfSBjYXRjaCB7IHJldHVybiByZXMud3JpdGVIZWFkKDQwNCkuZW5kKCkgfQogIHJlcy53cml0ZUhlYWQoMjAwLCB7ICdDb250ZW50LVR5cGUnOiBNSU1FW2V4dG5hbWUoZildID8/ICdhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0nLCAnQ29udGVudC1MZW5ndGgnOiBzIH0pCiAgY3JlYXRlUmVhZFN0cmVhbShmKS5waXBlKHJlcykKfSkKYXdhaXQgbmV3IFByb21pc2UoKHIpID0+IHNlcnZlci5saXN0ZW4oODkxNywgJzEyNy4wLjAuMScsIHIpKQpjb25zdCBicm93c2VyID0gYXdhaXQgcHVwcGV0ZWVyLmxhdW5jaCh7IGhlYWRsZXNzOiAnbmV3JywgYXJnczogWwogICctLW5vLXNhbmRib3gnLCAnLS11c2UtYW5nbGU9dnVsa2FuJywgJy0tZW5hYmxlLWZlYXR1cmVzPVZ1bGthbicsICctLWRpc2FibGUtdnVsa2FuLXN1cmZhY2UnLAogICctLWVuYWJsZS11bnNhZmUtd2ViZ3B1JywgJy0tZW5hYmxlLWRhd24tZmVhdHVyZXM9YWxsb3dfdW5zYWZlX2FwaXMsZGlzYWJsZV9hZGFwdGVyX2Jsb2NrbGlzdCcsCiAgJy0tZGlzYWJsZS1kYXduLWZlYXR1cmVzPXRpbWVzdGFtcF9xdWFudGl6YXRpb24nLCAnLS1pZ25vcmUtZ3B1LWJsb2NrbGlzdCcgXSwgcHJvdG9jb2xUaW1lb3V0OiA2MDAwMDAgfSkKY29uc3QgcGFnZSA9IGF3YWl0IGJyb3dzZXIubmV3UGFnZSgpCnBhZ2Uub24oJ2NvbnNvbGUnLCAobSkgPT4geyBjb25zdCB0ID0gbS50ZXh0KCk7IGlmICh0LnN0YXJ0c1dpdGgoJ1ttYl0nKSkgY29uc29sZS5sb2codCkgfSkKLy8gQ2hyb21lIG9uIExpbnV4L05WSURJQSAoVnVsa2FuKSBtYXkgd2l0aGhvbGQgc2hhZGVyLWYxNiBldmVuIHdoZW4gdGhlIGRyaXZlciBoYXMgaXQ7Ci8vIHRoZSBwYWdlIHRoZW4gcnVucyBhbiBmMzIgdHJhbnNjcmlwdGlvbiBvZiB0aGUgc2FtZSBrZXJuZWwgKD9mMzI9MSkgYW5kIHNheXMgc28uCmxldCBSCmZvciAoY29uc3QgcSBvZiBbJycsICc/ZjMyPTEnXSkgewogIGF3YWl0IHBhZ2UuZ290bygnaHR0cDovLzEyNy4wLjAuMTo4OTE3L21pY3JvYmVuY2guaHRtbCcgKyBxLCB7IHdhaXRVbnRpbDogJ2xvYWQnIH0pCiAgYXdhaXQgcGFnZS53YWl0Rm9yRnVuY3Rpb24oJ3dpbmRvdy5fX21iRG9uZSA9PT0gdHJ1ZScsIHsgdGltZW91dDogNjAwMDAwLCBwb2xsaW5nOiA1MDAgfSkKICBSID0gYXdhaXQgcGFnZS5ldmFsdWF0ZSgoKSA9PiB3aW5kb3cuX19tYikKICBpZiAoIShSLmVycm9yICYmIC9sYWNrcyBzaGFkZXItZjE2Ly50ZXN0KFIuZXJyb3IpKSkgYnJlYWsKICBjb25zb2xlLmxvZygnbm8gc2hhZGVyLWYxNiBvbiB0aGlzIGFkYXB0ZXI7IHJlcnVubmluZyBhcyBmMzInKQp9ClIuZGF0ZSA9IG5ldyBEYXRlKCkudG9JU09TdHJpbmcoKQpSLmdwdUluZm8gPSBhd2FpdCBwYWdlLmV2YWx1YXRlKGFzeW5jICgpID0+IHsgY29uc3QgYSA9IGF3YWl0IG5hdmlnYXRvci5ncHUucmVxdWVzdEFkYXB0ZXIoKTsgcmV0dXJuIHsgdmVuZG9yOiBhPy5pbmZvPy52ZW5kb3IsIGFyY2hpdGVjdHVyZTogYT8uaW5mbz8uYXJjaGl0ZWN0dXJlLCBkZXZpY2U6IGE/LmluZm8/LmRldmljZSwgZGVzY3JpcHRpb246IGE/LmluZm8/LmRlc2NyaXB0aW9uLCBmZWF0dXJlczogWy4uLihhPy5mZWF0dXJlcyA/PyBbXSldIH0gfSkKd3JpdGVGaWxlU3luYygnbWljcm9iZW5jaC1jb2xhYi10NC1mMzIuanNvbicsIEpTT04uc3RyaW5naWZ5KFIsIG51bGwsIDIpKQpjb25zb2xlLmxvZygnQURBUFRFUicsIEpTT04uc3RyaW5naWZ5KFIuZ3B1SW5mbykpCmlmIChSLmVycm9yKSBjb25zb2xlLmxvZygnUEFHRSBFUlJPUicsIFIuZXJyb3IpCmNvbnNvbGUubG9nKCdzYXZlZCBtaWNyb2JlbmNoLWNvbGFiLXQ0LWYzMi5qc29uOyBjb3B5IGl0IGJhY2sgaW50byBvdXQvIG9mIHRoZSBsaXRlcnQtY2FwdHVyZSBmb2xkZXInKQphd2FpdCBicm93c2VyLmNsb3NlKCk7IHNlcnZlci5jbG9zZSgpCg=="}
os.makedirs('kernel0112/out/shaders', exist_ok=True)
for name, data in files.items():
    with open(os.path.join('kernel0112', name), 'wb') as f:
        f.write(base64.b64decode(data))
!cd kernel0112 && npm init -y > /dev/null && npm i --silent puppeteer@25.11.0 > /dev/null && echo installed

In [ ]:
# 3) Run. Prints one line per variant; the JSON is the record to copy back.
!cd kernel0112 && node run.mjs
print(open('kernel0112/microbench-colab-t4-f32.json').read()[:600])

## Reading the result

`orig` is Google's kernel as shipped (16 × 4 workgroups, 12,288 threads). `wg256-same` changes only the workgroup size (control). `split16-64` and `split32-64` keep 64-thread workgroups and dispatch 4x and 8x the threads. The GB/s on each printed line is effective weight-streaming bandwidth: the 4.5 MiB (4,718,592 bytes) of weights the kernel reads per unit of kernel time, which is not a measurement of memory traffic, since a byte served from cache counts the same. Byte counts in this notebook are MiB (2^20 bytes) and GB/s is decimal. On the M2 Max: orig 60.9 µs, control 60.6, split32 36.8 (1.65x, in two runs whose browser, Chrome 146, is inferred from the adapter string, confirmed by provenanced out/microbench-0112-2026-09-24T07-36-19-655Z.json at 1.638x; same single-dispatch harness reads 0.84x in one recorded Chrome 131 run, 78.12 vs 93.12 us, on the same machine). Whether the T4 shows the same shape is the question this notebook answers. Download `kernel0112/microbench-colab-t4-f32.json` and drop it into `out/`.